# Load network and generate samples

In [ ]:
import keras
from keras import layers
import numpy as np
import tensorflow as tf
import subprocess
import matplotlib.pyplot as plt
import subprocess
import pandas as pd 
from _GAN_utils import SelfAttention3D

def build_generator():
    try:
        def get_gpu_memory_usage():
            command = "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits"
            memory_usage = subprocess.check_output(command, shell=True).decode("utf-8").strip().split("\n")
            memory_usage = [int(memory) for memory in memory_usage]
            return memory_usage
        mem_usage = get_gpu_memory_usage()
        # Allow memory growth to prevent TensorFlow from allocating all GPU memory at once
        gpus = tf.config.experimental.list_physical_devices('GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        # Specify which GPU to use for this script
        # For example, to use GPU 1: tf.config.experimental.set_visible_devices(gpus[1], 'GPU')
        tf.config.experimental.set_visible_devices(gpus[np.argmin(mem_usage)], 'GPU')
    except Exception as e:
        print(f"Can't select tf device: {e} \n")

    ###### build generator  ###########
    latent_dim = 64; channels=8
    gen_input_latent = keras.Input(shape=(latent_dim,))
    x = layers.Dense(4 * 4 * 4,activation="relu")(gen_input_latent)
    x = layers.Reshape((4, 4, 4, 1))(x)
    x = layers.Conv3D(16, 4, activation="relu", padding="same")(x)
    x = layers.Conv3DTranspose(32, 6, strides=2,activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv3D(32, 6, activation="relu", padding="same")(x)
    x = SelfAttention3D(32)(x)           
    x = layers.Conv3DTranspose(32, 8, 2,activation="relu", padding="same")(x)
    x = layers.Conv3D(32, 3,activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv3D(8, 3,activation="relu", padding="same")(x)
    x = layers.Conv3D(channels, 3, activation='sigmoid', padding="same")(x)
    return keras.models.Model(gen_input_latent, x,name="generator")

2025-10-24 09:38:18.021038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-24 09:38:18.048859: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-24 09:38:18.057184: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-24 09:38:18.082388: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-24 09:38:19.940847: W tensorflow/compiler/tf2

In [ ]:
n_generated = 30000
BATCH_SIZE = 512
th = 0.22

tag = "2025-09-14_10_05_29_WGAN_GP_corsika_NotConditioned_exceptfirst4_weigdecay_bigkernel"
orario = "2025-09-14_10_05_29"
epoca = 2250
 
generator_WGAN_AC = build_generator()


generator_WGAN_AC.load_weights(f"{tag}/generator_{tag}_{epoca}.keras")


random_latent_vectors = tf.random.normal(shape=(n_generated//BATCH_SIZE*BATCH_SIZE, 64))
generated_images = generator_WGAN_AC.predict(random_latent_vectors,BATCH_SIZE)

npart = 10**(generated_images*3.9) - 1
npnpart = np.array(npart)
npnpart_clip = np.where( npnpart <th , 0, npnpart)
npnpart_clip = np.where( (npnpart_clip >=th) & (npnpart_clip <1) , 1, npnpart_clip)


58/58 ━━━━━━━━━━━━━━━━━━━━ 8s 113ms/step
